# Inferência — PA1, segmentação de instâncias

Recebe o **caminho de uma imagem qualquer** e devolve a **máscara de instâncias
colorida** e a **contagem** de núcleos.

Não treina nada: carrega o checkpoint final (`outputs/p2/best.pth`, Trilha C, 100
épocas). A imagem pode estar em qualquer tamanho e em cor ou tons de cinza — ela é
levada para 256×256, que é a resolução de treino, e o mapa de rótulos volta no
tamanho original.

A lógica fica em `src/inference/predict.py`, que é coberto por testes; aqui só a
chamada e a figura.


## 1. Carregar o modelo (uma vez)


In [ ]:
import os, sys
from pathlib import Path

# funciona rodando de notebooks/ ou da raiz do repositório
RAIZ = Path.cwd()
if RAIZ.name == 'notebooks':
    RAIZ = RAIZ.parent
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

import matplotlib.pyplot as plt
from src.inference import carregar_modelo, prever
from src.utils import colorir

modelo, cfg, device = carregar_modelo(
    config='configs/p2_dsb2018.yaml',
    checkpoint='outputs/p2/best.pth',   # modelo final: Trilha C, 100 épocas
)
print('modelo carregado em', device)

## 2. Escolher a imagem e prever

Troque `CAMINHO` por qualquer arquivo de imagem. O padrão pega a primeira do
DSB2018 só para o notebook rodar sem configuração.


In [ ]:
import glob

CAMINHO = sorted(glob.glob('src/data/datasets/dsb2018/**/images/*.png', recursive=True))[0]
# CAMINHO = '/caminho/para/a/sua/imagem.png'

resultado = prever(CAMINHO, modelo, cfg)
print(f"{resultado['contagem']} núcleos encontrados")
print('arquivo:', CAMINHO)
print('tamanho da máscara:', resultado['rotulos'].shape)

## 3. Ver o resultado


In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(13, 4.5))

eixos[0].imshow(resultado['imagem'], cmap='gray')
eixos[0].set_title('imagem')

eixos[1].imshow(resultado['probabilidade'], cmap='magma', vmin=0, vmax=1)
eixos[1].set_title('probabilidade de foreground (256x256)')

eixos[2].imshow(colorir(resultado['rotulos']))
eixos[2].set_title(f"{resultado['contagem']} instâncias")

for e in eixos:
    e.axis('off')
plt.tight_layout()
plt.show()

## 4. Só a contagem, para várias imagens

O modelo já está carregado, então cada imagem extra custa só uma passada.


In [ ]:
for caminho in sorted(glob.glob('src/data/datasets/dsb2018/**/images/*.png', recursive=True))[:5]:
    r = prever(caminho, modelo, cfg)
    print(f"{r['contagem']:4d} núcleos  {Path(caminho).name[:24]}...")